# EVASPA Evaporative Fraction processing

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import ListedColormap

In [ ]:
from evaspa.ef import EFModel, check_variability, initialize, run
from evaspa.filter import determine_valid_pixels
from evaspa.io import read_data_from_file

In [ ]:
if os.environ["EVASPA_TEST_DATA_PATH"] is None:
    msg = "Variable EVASPA_TEST_DATA_PATH must be set"
    raise ValueError(msg)

### Get test data 

In [ ]:
file_path = os.path.join(
    os.environ["EVASPA_TEST_DATA_PATH"], "Landsat_20230327_28PCA.tif"
)

In [ ]:
data = read_data_from_file(file_path)
data

In [ ]:
coords = dict(data.sizes, band=3)
coords

In [ ]:
def plot_rgb(data: xr.Dataset, bands: list[str] | None = None):
    if bands is None:
        bands = ["red", "green", "blue"]
    rgb = xr.DataArray(
        np.dstack(
            (data[bands[0]].data, data[bands[1]].data, data[bands[2]].data)
        ),
        data.coords.assign(band=["r", "g", "b"]),
        dict(data.sizes, band=3),
    )
    rgb.plot.imshow(  # type: ignore
        rgb="band",
        vmin=rgb.quantile(0.01),
        vmax=rgb.quantile(0.99),
    )
    plt.title("RGB")


def plot_band(data: xr.Dataset, band: str):
    data[band].plot.imshow(  # type: ignore
        vmin=data[band].quantile(0.01),
        vmax=data[band].quantile(0.99),
    )


def plot_mask(data: xr.Dataset, mask: str, invert=False):
    cmap = [(0.0, 0.0, 0.0, 0.0)]
    if len(np.unique(data[mask].data)) > 1:
        cmap = [(1.0, 0.0, 0.0, 1.0), (0.0, 0.0, 0.0, 0.0)]
    if invert:
        cmap = cmap[::-1]
    valid_cmap = ListedColormap(cmap)
    data[mask].plot(cmap=valid_cmap, add_colorbar=False)


def plot(
    data: xr.Dataset,
    var_name: str | None = None,
    model: EFModel | None = None,
    save: bool = False,
):
    """
    Plot
    """
    # dimensions
    # gridspec inside gridspec
    fig = plt.figure(constrained_layout=True, figsize=(4, 3))

    ax = plt.gca()

    if var_name is None and model is None:
        msg = "Provide either var_name or model"
        raise ValueError(msg)
    lst = data["lst"].data

    if var_name is not None:
        var = data[var_name].data
        # plot all points
        ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)

    # plot edges
    elif model is not None:
        var_name = model.var
        var = data[model.var].data
        # plot all points
        ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)
        # edges coordinates
        var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
        var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)

        var_arr = np.linspace(var_min, var_max, num=20)
        tw_arr = model.twet(var_arr)
        td_arr = model.tdry(var_arr)

        ax.plot(
            var_arr,
            td_arr,
            color="tab:red",
            linewidth=3,
            label="dry edge",
            zorder=1,
        )
        ax.plot(
            var_arr,
            tw_arr,
            color="tab:blue",
            linewidth=3,
            label="wet edge",
            zorder=1,
        )

    # Labels
    ax.set_xlabel(var_name)
    ax.set_ylabel("Land Surface Temperature K")

    # title
    ax.set_title("Evaporative fraction method", fontsize=12)

    # savefig
    if save:
        destfile = "ef.png"
        fig.savefig(destfile)

    plt.show()

In [ ]:
plot_rgb(data)

In [ ]:
plot_band(data, "lst")

In [ ]:
plot(data, "albedo")

In [ ]:
plot(data, "lai")

### Configuration

In [ ]:
config = {
    "models": [
        {
            "name": "model1",
            "dry_edge": {
                "type": "LinearEdge",
                "config": {
                    "interval_type": "size",
                    "interval_nb": 20,
                    "percentile": [98, 100],
                    "selection": "median",
                },
            },
            "wet_edge": {
                "type": "LinearEdge",
                "config": {
                    "interval_type": "size",
                    "interval_nb": 20,
                    "percentile": [0, 2],
                    "selection": "median",
                },
            },
            "var": "albedo",
        },
        {
            "name": "model2",
            "dry_edge": {
                "type": "LinearEdge",
                "config": {
                    "interval_type": "density",
                    "interval_nb": 20,
                    "percentile": [98, 100],
                    "selection": "median",
                },
            },
            "wet_edge": {
                "type": "FlatEdge",
                "config": {"selection": "min"},
            },
            "var": "lai",
        },
    ],
    "options": {
        "selection": False,
        "merging": "mean",
    },
}

In [ ]:
pprint(config, sort_dicts=False)

### Process evaporative fraction

#### Filter

In [ ]:
data["valid"] = determine_valid_pixels(data, cloud="cloud")

In [ ]:
plot_mask(data=data, mask="valid", invert=True)

#### Check variability

In [ ]:
check_variability(data["lst"], mask=data["valid"], threshold=0.1)

#### Compute EF

In [ ]:
models, options = initialize(config)

In [ ]:
ef, ef_inst = run(models, data, mask=None, **options)

In [ ]:
ef

In [ ]:
plot_band(ef_inst, band="ef")

In [ ]:
plot(data, model=EFModel.create(ef["model1"].attrs))

In [ ]:
plot(data, model=EFModel.create(ef["model2"].attrs))